# Assignment 8 - Cache Mapping: Direct Mapped vs 2-Way Set Associative

In [ ]:
import math

# System Configuration
main_mem_size = 1 * 1024 * 1024 * 1024   # 1 GB
cache_mem_size = 64 * 1024               # 64 KB
block_size = 4                           # 4 Bytes

# Bit-width & Field Calculations
address_bits = int(math.log2(main_mem_size))   # 30 bits
offset_bits = int(math.log2(block_size))       # 2 bits

# Direct Mapped (1-Way) Calculations
num_lines_direct = cache_mem_size // block_size
index_bits_direct = int(math.log2(num_lines_direct))
tag_bits_direct = address_bits - index_bits_direct - offset_bits

# 2-Way Set Associative Calculations
# Set size = 2 lines per set. Total sets = Cache lines / 2
num_lines_total = cache_mem_size // block_size
num_sets_2way = num_lines_total // 2
index_bits_2way = int(math.log2(num_sets_2way))
tag_bits_2way = address_bits - index_bits_2way - offset_bits

print("=" * 60)
print("                  CACHE SYSTEM PARAMETERS                  ")
print("=" * 60)
print(f"Physical Address Width  : {address_bits} bits")
print(f"Total Cache Lines       : {num_lines_total:,}")
print(f"Offset Field Width      : {offset_bits} bits")
print("-" * 60)
print("1. DIRECT MAPPED (ONE-WAY ASSOCIATIVE) FIELDS:")
print(f"   - Index (Line Number) : {index_bits_direct} bits ({num_lines_direct:,} lines)")
print(f"   - Tag Field           : {tag_bits_direct} bits")
print("-" * 60)
print("2. TWO-WAY SET ASSOCIATIVE FIELDS:")
print(f"   - Index (Set Number)  : {index_bits_2way} bits ({num_sets_2way:,} sets)")
print(f"   - Tag Field           : {tag_bits_2way} bits")
print("=" * 60)

## One-Way Set Associative (Direct Mapped)
Address breakdown: `[ 14-bit Tag | 14-bit Line No (Index) | 2-bit Offset ]`

In [ ]:
class DirectMappedCache:
    def __init__(self, num_lines, block_size, address_bits):
        self.num_lines = num_lines
        self.block_size = block_size
        self.address_bits = address_bits
        # Initialize cache rows with valid bit (0/1) and tag store
        self.cache = [{"valid": 0, "tag": 0} for _ in range(self.num_lines)]

        # Bit-widths
        self.offset_bits = int(math.log2(self.block_size))
        self.index_bits = int(math.log2(self.num_lines))
        self.tag_bits = self.address_bits - self.index_bits - self.offset_bits

    def access(self, address):
        # Mask address to simulated physical limit
        addr_masked = address & ((1 << self.address_bits) - 1)
        # Extract fields
        offset = addr_masked & ((1 << self.offset_bits) - 1)
        index = (addr_masked >> self.offset_bits) & ((1 << self.index_bits) - 1)
        tag = (addr_masked >> (self.offset_bits + self.index_bits)) & ((1 << self.tag_bits) - 1)

        line = self.cache[index]
        is_hit = (line["valid"] == 1) and (line["tag"] == tag)
        if not is_hit:
            # Fetch block & overwrite matching line slot
            line["valid"] = 1
            line["tag"] = tag
        status = "HIT" if is_hit else "MISS"
        return status, tag, index, offset

In [ ]:
# Instantiate Direct Mapped Cache
direct_cache = DirectMappedCache(num_lines=16384, block_size=4, address_bits=30)

# Test Addresses
test_addresses = [0, 4, 8, 12, 16, 16384, 0, 32768, 4, 16388, 32, 16384]

print("=" * 75)
print("                       DIRECT MAPPED CACHE SIMULATION                       ")
print("=" * 75)
print(f"{'Address':<10} | {'Tag (Dec)':<10} | {'Line Index (Dec)':<16} | {'Offset':<8} | {'Result':<6}")
print("-" * 75)

d_hits, d_misses = 0, 0
for addr in test_addresses:
    status, tag, index, offset = direct_cache.access(addr)
    if status == "HIT":
        d_hits += 1
    else:
        d_misses += 1
    print(f"{addr:<10} | {tag:<10} | {index:<16} | {offset:<8} | {status:<6}")

total_d = len(test_addresses)
print("-" * 75)
print(f"Total Accesses: {total_d} | Hits: {d_hits} | Misses: {d_misses} | Hit Rate: {(d_hits/total_d)*100:.2f}%")
print("=" * 75)

## Two-Way Set Associative Cache
Address breakdown: `[ 15-bit Tag | 13-bit Set No (Index) | 2-bit Offset ]`
Eviction policy: LRU tracked per set.

In [ ]:
class TwoWaySetAssociativeCache:
    def __init__(self, num_sets, block_size, address_bits):
        self.num_sets = num_sets
        self.block_size = block_size
        self.address_bits = address_bits
        # Each Set contains 2 lines, each track valid, tag, and an lru counter
        self.cache = [[
            {"valid": 0, "tag": 0, "lru": 0},
            {"valid": 0, "tag": 0, "lru": 0}
        ] for _ in range(self.num_sets)]

        self.offset_bits = int(math.log2(self.block_size))
        self.index_bits = int(math.log2(self.num_sets))
        self.tag_bits = self.address_bits - self.index_bits - self.offset_bits

    def access(self, address):
        addr_masked = address & ((1 << self.address_bits) - 1)
        # Extract fields
        offset = addr_masked & ((1 << self.offset_bits) - 1)
        set_index = (addr_masked >> self.offset_bits) & ((1 << self.index_bits) - 1)
        tag = (addr_masked >> (self.offset_bits + self.index_bits)) & ((1 << self.tag_bits) - 1)

        cache_set = self.cache[set_index]
        is_hit = False
        hit_way = -1

        # Check ways in current Set
        for way in range(2):
            if cache_set[way]["valid"] == 1 and cache_set[way]["tag"] == tag:
                is_hit = True
                hit_way = way
                break

        if is_hit:
            # Promote hit way to MRU (increasing LRU tracker index value)
            cache_set[hit_way]["lru"] = max(cache_set[0]["lru"], cache_set[1]["lru"]) + 1
            status = "HIT"
        else:
            status = "MISS"
            # Identify entry to evict based on empty slot or lowest LRU counter value
            if cache_set[0]["valid"] == 0:
                target_way = 0
            elif cache_set[1]["valid"] == 0:
                target_way = 1
            else:
                # Both slots filled, evict way with lower LRU value
                target_way = 0 if cache_set[0]["lru"] < cache_set[1]["lru"] else 1

            # Load new Block data into target way
            cache_set[target_way]["valid"] = 1
            cache_set[target_way]["tag"] = tag
            cache_set[target_way]["lru"] = max(cache_set[0]["lru"], cache_set[1]["lru"]) + 1

        return status, tag, set_index, offset

In [ ]:
# Instantiate Two-Way Cache (16,384 total lines / 2 ways = 8,192 sets)
tw_cache = TwoWaySetAssociativeCache(num_sets=8192, block_size=4, address_bits=30)

test_addresses = [0, 4, 8, 12, 16, 16384, 0, 32768, 4, 16388, 32, 16384]

print("=" * 75)
print("                    TWO-WAY ASSOCIATIVE CACHE SIMULATION                    ")
print("=" * 75)
print(f"{'Address':<10} | {'Tag (Dec)':<10} | {'Set Index (Dec)':<16} | {'Offset':<8} | {'Result':<6}")
print("-" * 75)

tw_hits, tw_misses = 0, 0
for addr in test_addresses:
    status, tag, index, offset = tw_cache.access(addr)
    if status == "HIT":
        tw_hits += 1
    else:
        tw_misses += 1
    print(f"{addr:<10} | {tag:<10} | {index:<16} | {offset:<8} | {status:<6}")

total_tw = len(test_addresses)
print("-" * 75)
print(f"Total Accesses: {total_tw} | Hits: {tw_hits} | Misses: {tw_misses} | Hit Rate: {(tw_hits/total_tw)*100:.2f}%")
print("=" * 75)